# Notebook 01 — Tests du domaine métier

Ce notebook sert à **comprendre et vérifier les objets métier de OrderOps** définis dans `src/orderops/domain/`.

Il fonctionne **sans infrastructure externe** : aucune base PostgreSQL, aucun serveur MCP et aucun appel à Mistral ne sont nécessaires.

Les objectifs sont de vérifier que :

- tous les modèles et énumérations du domaine sont importables ;
- les objets métier peuvent être créés avec des données valides ;
- les contraintes Pydantic sont respectées ;
- `CreateIncidentCommand` normalise correctement ses données ;
- `request_hash()` est stable pour un même contenu métier ;
- un contenu métier différent produit un hash différent ;
- les modèles du domaine sont immuables.

Les expériences suivent **Arrange → Act → Assert**.  
Le notebook sert à **comprendre** ; les tests durables sont ensuite conservés dans `tests/unit/`.


## 1. Imports du domaine

On importe les dépendances et tous les objets métier utilisés dans le notebook.  
Si cette cellule fonctionne, les principaux modules du domaine sont bien importables.


In [ ]:
from datetime import UTC, date, datetime
from decimal import Decimal

import ipytest
import pytest
from pydantic import ValidationError

from orderops.domain.enums import DeliveryStatus, IncidentStatus, OrderStatus
from orderops.domain.models import (
    CreateIncidentCommand,
    Customer,
    Delivery,
    Incident,
    IncidentCreationResult,
    Order,
    OrderItem,
    ProductStock,
)

ipytest.autoconfig()
print("Imports du domaine : OK")


## 2. Vérification des statuts métier

Les énumérations imposent un vocabulaire commun pour les statuts utilisés par l'application.


In [ ]:
assert OrderStatus.PENDING == "pending"
assert OrderStatus.SHIPPED == "shipped"
assert DeliveryStatus.IN_TRANSIT == "in_transit"
assert DeliveryStatus.DELAYED == "delayed"
assert IncidentStatus.OPEN == "open"
assert IncidentStatus.RESOLVED == "resolved"

print("Enums du domaine : OK")


## 3. Création des principaux objets métier

On vérifie que chaque modèle principal peut être créé avec des données valides.


In [ ]:
customer = Customer(
    customer_id="CUS-001",
    name="ACME",
    email="sav@acme.example",
    phone="+33100000001",
)
customer


In [ ]:
item = OrderItem(
    product_id="SKU-12",
    product_name="Scanner mobile",
    quantity=2,
    unit_price=Decimal("249.90"),
)
item


In [ ]:
order = Order(
    order_id="CMD-1042",
    customer_id="CUS-001",
    status=OrderStatus.SHIPPED,
    created_at=datetime.now(UTC),
    expected_delivery=date(2026, 8, 18),
    items=(item,),
)
order


In [ ]:
delivery = Delivery(
    delivery_id="DEL-1042",
    order_id="CMD-1042",
    status=DeliveryStatus.DELAYED,
    carrier="Demo Transport",
    tracking_number="TRACK-1042",
    expected_delivery=date(2026, 8, 18),
)
delivery


In [ ]:
stock = ProductStock(
    product_id="SKU-12",
    product_name="Scanner mobile",
    on_hand_quantity=20,
    reserved_quantity=4,
    quantity_available=16,
)
stock


In [ ]:
incident = Incident(
    incident_id="INC-001",
    order_id="CMD-1042",
    reason="Livraison en retard",
    status=IncidentStatus.OPEN,
    idempotency_key="incident-CMD-1042",
    created_at=datetime.now(UTC),
)
incident


In [ ]:
incident_result = IncidentCreationResult(
    incident=incident,
    already_existed=False,
)
incident_result


## 4. Contraintes Pydantic

On vérifie que les données invalides sont refusées dès la création des objets.


In [ ]:
with pytest.raises(ValidationError):
    OrderItem(
        product_id="SKU-12",
        product_name="Scanner mobile",
        quantity=0,
        unit_price=Decimal("249.90"),
    )

with pytest.raises(ValidationError):
    OrderItem(
        product_id="SKU-12",
        product_name="Scanner mobile",
        quantity=2,
        unit_price=Decimal("-10.00"),
    )

with pytest.raises(ValidationError):
    ProductStock(
        product_id="SKU-12",
        product_name="Scanner mobile",
        on_hand_quantity=-1,
        reserved_quantity=0,
        quantity_available=0,
    )

print("Contraintes Pydantic : OK")


## 5. Normalisation de `CreateIncidentCommand`

Le modèle supprime les espaces inutiles autour des identifiants et normalise les espaces dans `reason`.


In [ ]:
command = CreateIncidentCommand(
    order_id=" CMD-1042 ",
    reason="Retard   confirmé   par le transporteur",
    idempotency_key=" notebook-CMD-1042 ",
)

assert command.order_id == "CMD-1042"
assert command.reason == "Retard confirmé par le transporteur"
assert command.idempotency_key == "notebook-CMD-1042"

command.model_dump()


## 6. Stabilité du `request_hash()`

Même contenu métier → même hash.  
Contenu métier différent → hash différent.  
La clé d'idempotence n'entre volontairement pas dans le hash.


In [ ]:
same_payload = CreateIncidentCommand(
    order_id="CMD-1042",
    reason="Retard confirmé par le transporteur",
    idempotency_key="another-key-1042",
)

different_payload = CreateIncidentCommand(
    order_id="CMD-1042",
    reason="Colis déclaré perdu par le transporteur",
    idempotency_key="notebook-CMD-1042",
)

assert len(command.request_hash()) == 64
assert same_payload.request_hash() == command.request_hash()
assert different_payload.request_hash() != command.request_hash()

print("Hash :", command.request_hash())


## 7. Rejet des commandes invalides

Une commande qui ne respecte pas les contraintes de longueur doit provoquer une `ValidationError`.


In [ ]:
with pytest.raises(ValidationError):
    CreateIncidentCommand(
        order_id="X",
        reason="non",
        idempotency_key="courte",
    )

print("Commande invalide refusée : OK")

## 8. Immutabilité des objets du domaine

Tous les modèles héritent de `DomainModel` avec `frozen=True`.  
Une fois créé, un objet métier ne peut pas être modifié directement.


In [ ]:
with pytest.raises(ValidationError):
    customer.name = "Nouveau nom"

print("Immutabilité : OK")


## 9. Tests unitaires avec `ipytest`

Les observations précédentes sont maintenant exprimées sous forme de tests Pytest exécutables dans le notebook.

Les versions durables de ces tests doivent être placées dans `tests/unit/test_domain.py`.


In [ ]:
%%ipytest -q

def test_order_item_rejects_zero_quantity() -> None:
    with pytest.raises(ValidationError):
        OrderItem(
            product_id="SKU-12",
            product_name="Scanner mobile",
            quantity=0,
            unit_price=Decimal("249.90"),
        )


def test_command_normalizes_values() -> None:
    command = CreateIncidentCommand(
        order_id=" CMD-1042 ",
        reason="Retard   confirmé par le transporteur",
        idempotency_key=" notebook-CMD-1042 ",
    )

    assert command.order_id == "CMD-1042"
    assert command.reason == "Retard confirmé par le transporteur"
    assert command.idempotency_key == "notebook-CMD-1042"


def test_same_payload_produces_same_hash() -> None:
    first = CreateIncidentCommand(
        order_id="CMD-1042",
        reason="Livraison en retard",
        idempotency_key="first-key",
    )
    second = CreateIncidentCommand(
        order_id="CMD-1042",
        reason="Livraison en retard",
        idempotency_key="second-key",
    )

    assert first.request_hash() == second.request_hash()


def test_invalid_command_is_rejected() -> None:
    with pytest.raises(ValidationError):
        CreateIncidentCommand(
            order_id="X",
            reason="non",
            idempotency_key="courte",
        )


def test_domain_model_is_immutable() -> None:
    customer = Customer(
        customer_id="CUS-001",
        name="ACME",
        email="sav@acme.example",
    )

    with pytest.raises(ValidationError):
        customer.name = "Autre nom"


## Checkpoint du Notebook 01

Le domaine est validé lorsque :

- tous les modèles et enums sont importables ;
- les objets valides sont créés correctement ;
- les données invalides sont refusées ;
- la normalisation fonctionne ;
- le hash est stable ;
- les modèles sont immuables ;
- les tests `ipytest` passent ;
- aucune infrastructure externe n'a été utilisée.
